In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
file_path = '/content/drive/MyDrive/vinuni_datathon2026/model/processed_data.csv'

df = pd.read_csv(file_path)

display(df.head())

,date,order_id,customer_id,order_status_cancelled,order_status_created,order_status_delivered,order_status_paid,order_status_returned,order_status_shipped,payment_method_apple_pay,...,day_name,traffic_direct,traffic_email_campaign,traffic_organic_search,traffic_paid_search,traffic_referral,traffic_social_media,is_web_data_simulated,Revenue,COGS
0,2012-07-04,162,161,9,0,134,6,11,2,19,...,NaN,0,0,0,0,0,0,1,5123547.94,3982991.19
1,2012-07-05,97,97,9,0,81,1,5,1,7,...,NaN,0,0,0,0,0,0,1,2751773.45,2150580.23
2,2012-07-06,93,93,11,1,69,2,7,3,8,...,NaN,0,0,0,0,0,0,1,3054029.42,2517632.84
3,2012-07-07,73,73,8,3,55,0,6,1,10,...,NaN,0,0,0,0,0,0,1,2667930.94,2108246.62
4,2012-07-08,88,87,5,3,67,2,8,3,12,...,NaN,0,0,0,0,0,0,1,2360851.90,1808622.79


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 94 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   date                          3833 non-null   object 
 1   order_id                      3833 non-null   int64  
 2   customer_id                   3833 non-null   int64  
 3   order_status_cancelled        3833 non-null   int64  
 4   order_status_created          3833 non-null   int64  
 5   order_status_delivered        3833 non-null   int64  
 6   order_status_paid             3833 non-null   int64  
 7   order_status_returned         3833 non-null   int64  
 8   order_status_shipped          3833 non-null   int64  
 9   payment_method_apple_pay      3833 non-null   int64  
 10  payment_method_bank_transfer  3833 non-null   int64  
 11  payment_method_cod            3833 non-null   int64  
 12  payment_method_credit_card    3833 non-null   int64  
 13  pay

In [ ]:
import holidays

# Convert 'date' column to datetime
df['date'] = pd.to_datetime(df['date'])

# Extract 'is_weekend'
df['is_weekend'] = df['date'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)

# Extract 'quarter'
df['quarter'] = df['date'].dt.quarter

# Define the date range for holidays
start_date = pd.to_datetime('2012-07-04')
end_date = pd.to_datetime('2022-12-31')

# Get Vietnamese holidays within the specified range
vn_holidays = holidays.country_holidays('VN', years=range(start_date.year, end_date.year + 1))

# Create a list of holiday dates, converting Timestamps to date objects for comparison
holiday_dates = [date for date in vn_holidays if start_date.date() <= date <= end_date.date()]

# Add a new column 'is_holiday'
df['is_holiday'] = df['date'].isin(holiday_dates).astype(int)

display(df[['date', 'is_weekend', 'quarter', 'is_holiday']].head())

,date,is_weekend,quarter,is_holiday
0,2012-07-04,0,3,0
1,2012-07-05,0,3,0
2,2012-07-06,0,3,0
3,2012-07-07,1,3,0
4,2012-07-08,1,3,0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3833 entries, 0 to 3832
Data columns (total 97 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   date                          3833 non-null   datetime64[ns]
 1   order_id                      3833 non-null   int64         
 2   customer_id                   3833 non-null   int64         
 3   order_status_cancelled        3833 non-null   int64         
 4   order_status_created          3833 non-null   int64         
 5   order_status_delivered        3833 non-null   int64         
 6   order_status_paid             3833 non-null   int64         
 7   order_status_returned         3833 non-null   int64         
 8   order_status_shipped          3833 non-null   int64         
 9   payment_method_apple_pay      3833 non-null   int64         
 10  payment_method_bank_transfer  3833 non-null   int64         
 11  payment_method_cod            

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# 1. Đảm bảo dữ liệu được sắp xếp và tạo biến đếm thời gian (time_idx) để học Trend
df = df.sort_values('date').reset_index(drop=True)
df['time_idx'] = np.arange(len(df))

# 2. Xây dựng Linear Regression để học Trend
# CHÚ Ý: Chỉ fit trên tập Train (trước 2021) để tránh việc rò rỉ dữ liệu (Data Leakage)
train_mask = df['date'] < '2021-01-01'

trend_model_rev = LinearRegression()
trend_model_rev.fit(df.loc[train_mask, ['time_idx']], df.loc[train_mask, 'Revenue'])

trend_model_cogs = LinearRegression()
trend_model_cogs.fit(df.loc[train_mask, ['time_idx']], df.loc[train_mask, 'COGS'])

# 3. Tính toán đường Trend và PHẦN DƯ (Residuals = Thực tế - Trend)
df['trend_rev'] = trend_model_rev.predict(df[['time_idx']])
df['trend_cogs'] = trend_model_cogs.predict(df[['time_idx']])

df['resid_rev'] = df['Revenue'] - df['trend_rev']
df['resid_cogs'] = df['COGS'] - df['trend_cogs']

# 4. TẠO LẠI CÁC BIẾN LAG VÀ ROLLING DỰA TRÊN "PHẦN DƯ" (VÔ CÙNG QUAN TRỌNG)
lags = [1, 7, 30]

for lag in lags:
    # Thay vì shift Revenue, ta shift phần dư (resid_rev)
    df[f'resid_rev_lag_{lag}'] = df['resid_rev'].shift(lag)
    df[f'resid_cogs_lag_{lag}'] = df['resid_cogs'].shift(lag)
    df[f'sessions_lag_{lag}'] = df['sessions'].shift(lag)

df['resid_rev_rolling_mean_7d'] = df['resid_rev'].shift(1).rolling(window=7).mean()
df['resid_cogs_rolling_mean_7d'] = df['resid_cogs'].shift(1).rolling(window=7).mean()

# Loại bỏ NaN
df = df.dropna(subset=[f'resid_rev_lag_{max(lags)}'])

In [ ]:
# CHIA DỮ LIỆU
# Đảm bảo cột date đúng định dạng
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')

# Chia dữ liệu
train = df[df['date'] < '2021-01-01']
valid = df[(df['date'] >= '2021-01-01') & (df['date'] < '2022-01-01')]
test = df[df['date'] >= '2022-01-01']

print(f"Train: {train.shape[0]} dòng | Valid: {valid.shape[0]} dòng | Test: {test.shape[0]} dòng")

Train: 3073 dòng | Valid: 365 dòng | Test: 365 dòng


In [ ]:
# Cập nhật loại bỏ các biến không dùng (bao gồm cả các biến gốc)
drop_cols = ['date', 'Revenue', 'COGS', 'day_name', 'trend_rev', 'trend_cogs', 'resid_rev', 'resid_cogs']
features = [col for col in df.columns if col not in drop_cols]

# CHIA LẠI DATA VỚI TARGET LÀ PHẦN DƯ (RESIDUALS)
X_train = train[features]
y_train_rev = train['resid_rev']
y_train_cogs = train['resid_cogs']

X_valid = valid[features]
y_valid_rev = valid['resid_rev']
y_valid_cogs = valid['resid_cogs']

X_test = test[features]

# --- TUNE LẠI THAM SỐ XGBOOST ---
from xgboost import XGBRegressor

# Để learning_rate nhỏ lại và thêm subsample để mô hình bền vững hơn
xgb_model_rev = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model_rev.fit(X_train, y_train_rev, eval_set=[(X_valid, y_valid_rev)])

xgb_model_cogs = XGBRegressor(
    n_estimators=1500,
    learning_rate=0.01,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model_cogs.fit(X_train, y_train_cogs, eval_set=[(X_valid, y_valid_cogs)])

In [ ]:
# 1. Linear Regression nhẩm tính đường Trend cho tập Test
test_trend_rev = trend_model_rev.predict(test[['time_idx']])
test_trend_cogs = trend_model_cogs.predict(test[['time_idx']])

# 2. XGBoost dự đoán phần dư (gợn sóng) trên tập Test
xgb_preds_resid_rev = xgb_model_rev.predict(X_test)
xgb_preds_resid_cogs = xgb_model_cogs.predict(X_test)

# 3. KẾT QUẢ CỘNG GỘP = TREND DÀI HẠN + PHẦN DƯ NGẮN HẠN
final_preds_rev = test_trend_rev + xgb_preds_resid_rev
final_preds_cogs = test_trend_cogs + xgb_preds_resid_cogs

# 4. Tính toán các chỉ số đánh giá (Metrics) với Target thực tế
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, r2_score

# Y thực tế vẫn lấy từ tập test gốc
y_test_rev_actual = test['Revenue']
y_test_cogs_actual = test['COGS']

# Metrics for Revenue
mae_rev = mean_absolute_error(y_test_rev_actual, final_preds_rev)
mape_rev = mean_absolute_percentage_error(y_test_rev_actual, final_preds_rev)
r2_rev = r2_score(y_test_rev_actual, final_preds_rev)

print(f"--- KẾT QUẢ TRÊN TẬP TEST (REVENUE) ---")
print(f"MAE (Sai số tuyệt đối trung bình): {mae_rev:,.0f} VNĐ")
print(f"MAPE (Phần trăm sai số): {mape_rev:.2%}")
print(f"R2 Score (Độ chính xác xu hướng): {r2_rev:.4f}")

# Metrics for COGS
mae_cogs = mean_absolute_error(y_test_cogs_actual, final_preds_cogs)
mape_cogs = mean_absolute_percentage_error(y_test_cogs_actual, final_preds_cogs)
r2_cogs = r2_score(y_test_cogs_actual, final_preds_cogs)

print(f"\n--- KẾT QUẢ TRÊN TẬP TEST (COGS) ---")
print(f"MAE (Sai số tuyệt đối trung bình): {mae_cogs:,.0f} VNĐ")
print(f"MAPE (Phần trăm sai số): {mape_cogs:.2%}")
print(f"R2 Score (Độ chính xác xu hướng): {r2_cogs:.4f}")

--- KẾT QUẢ TRÊN TẬP TEST (REVENUE) ---
MAE (Sai số tuyệt đối trung bình): 554,064 VNĐ
MAPE (Phần trăm sai số): 20.21%
R2 Score (Độ chính xác xu hướng): 0.8715

--- KẾT QUẢ TRÊN TẬP TEST (COGS) ---
MAE (Sai số tuyệt đối trung bình): 626,913 VNĐ
MAPE (Phần trăm sai số): 25.56%
R2 Score (Độ chính xác xu hướng): 0.7758


In [ ]:
last_date = df['date'].max()
last_time_idx = df['time_idx'].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), end='2024-07-01', freq='D')

df_forecast = df.copy()

for current_date in future_dates:
    last_row = df_forecast.iloc[-1:].copy()
    last_row['date'] = current_date

    # 1. Tăng time_idx để bắt Trend leo dốc trong tương lai
    last_time_idx += 1
    last_row['time_idx'] = last_time_idx

    # ... (Cập nhật is_weekend, quarter, is_holiday giống code cũ của bạn) ...
    last_row['is_weekend'] = 1 if current_date.dayofweek >= 5 else 0

    # 2. Linear Regression nhẩm tính giá trị Trend của tương lai
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pred_trend_rev = trend_model_rev.predict([[last_time_idx]])[0]
        pred_trend_cogs = trend_model_cogs.predict([[last_time_idx]])[0]

    # 3. Cập nhật Lag và Rolling dựa trên "Phần dư"
    for lag in [1, 7, 30]:
        last_row[f'resid_rev_lag_{lag}'] = df_forecast.iloc[-lag]['resid_rev'] if len(df_forecast) >= lag else df_forecast.iloc[-1]['resid_rev']
        last_row[f'resid_cogs_lag_{lag}'] = df_forecast.iloc[-lag]['resid_cogs'] if len(df_forecast) >= lag else df_forecast.iloc[-1]['resid_cogs']

    last_row['resid_rev_rolling_mean_7d'] = df_forecast['resid_rev'].tail(7).mean()
    last_row['resid_cogs_rolling_mean_7d'] = df_forecast['resid_cogs'].tail(7).mean()

    # 4. XGBoost dự đoán phần dư hiện tại
    current_features = last_row[features]
    pred_resid_rev = xgb_model_rev.predict(current_features)[0]
    pred_resid_cogs = xgb_model_cogs.predict(current_features)[0]

    # 5. TỔNG HỢP (KẾT QUẢ THỰC TẾ TRẢ RA = TREND + PHẦN DƯ)
    final_pred_rev = pred_trend_rev + pred_resid_rev
    final_pred_cogs = pred_trend_cogs + pred_resid_cogs

    last_row['Revenue'] = final_pred_rev
    last_row['COGS'] = final_pred_cogs
    last_row['trend_rev'] = pred_trend_rev
    last_row['trend_cogs'] = pred_trend_cogs
    last_row['resid_rev'] = pred_resid_rev
    last_row['resid_cogs'] = pred_resid_cogs

    df_forecast = pd.concat([df_forecast, last_row], ignore_index=True)

submission = df_forecast[df_forecast['date'] > '2022-12-31'][['date', 'Revenue', 'COGS']]
submission.to_csv('submission_tuned.csv', index=False)

In [ ]:
submission

,date,Revenue,COGS
3803,2023-01-01,1.542079e+06,1.177499e+06
3804,2023-01-02,1.464221e+06,1.064809e+06
3805,2023-01-03,1.461934e+06,1.052788e+06
3806,2023-01-04,1.460217e+06,1.051385e+06
3807,2023-01-05,1.461442e+06,1.058203e+06
...,...,...,...
4346,2024-06-27,1.139817e+06,7.996675e+05
4347,2024-06-28,1.139203e+06,7.991640e+05
4348,2024-06-29,1.139529e+06,7.986605e+05
4349,2024-06-30,1.138916e+06,7.981569e+05


In [ ]:
submission.info()

In [ ]:
# 1. Rename 'date' column to 'Date'
submission = submission.rename(columns={'date': 'Date'})

# 2. Convert 'Date' column to object (string) type
submission['Date'] = submission['Date'].dt.strftime('%Y-%m-%d')

# Display the info of the modified submission DataFrame to verify
submission.info()

# Display the head of the modified submission DataFrame
display(submission.head())

<class 'pandas.core.frame.DataFrame'>
Index: 548 entries, 3803 to 4350
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     548 non-null    object 
 1   Revenue  548 non-null    float64
 2   COGS     548 non-null    float64
dtypes: float64(2), object(1)
memory usage: 17.1+ KB


,Date,Revenue,COGS
3803,2023-01-01,1.542079e+06,1.177499e+06
3804,2023-01-02,1.464221e+06,1.064809e+06
3805,2023-01-03,1.461934e+06,1.052788e+06
3806,2023-01-04,1.460217e+06,1.051385e+06
3807,2023-01-05,1.461442e+06,1.058203e+06


DataFrame `submission` đã được định dạng chính xác. Bây giờ, tôi sẽ lưu nó vào một tệp CSV mới và cung cấp mã để bạn tải xuống.

In [ ]:
submission_file_name = 'formatted_submission_recursive_2023_2024.csv'
submission.to_csv(submission_file_name, index=False)

print(f"Submission file '{submission_file_name}' created successfully.")

# Download the formatted submission file
from google.colab import files
files.download(submission_file_name)

Submission file 'formatted_submission_recursive_2023_2024.csv' created successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files
files.download('formatted_submission_recursive_2023_2024.csv')